# 🔬 Model Pipeline: Microsoft Phi-3-Mini-4K-Instruct (4-bit QLoRA)
### **Jadavpur University Research Internship | Under: Prof. Tohida Rehman**
**Track:** FIRE MixSub-SciHigh Scientific Highlight Summarization
**Model ID:** `microsoft/Phi-3-mini-4k-instruct`
**Architecture:** CAUSAL (4-bit QLoRA)

---
### 🎯 Strict Uniform Benchmark Constraints (Scientifically Fair)
- **`MAX_SOURCE_LENGTH = 512` tokens**: Uniform input length across all 6 models.
- **`MAX_TARGET_LENGTH = 128` tokens**: Uniform output generation limit across all 6 models.

### 📊 The 3 Non-Redundant Benchmark Pillars Measured:
1. **Overall Quality**: ROUGE-L F1 (↑ Higher is Better) & Target F1 Score (↑)
2. **Hallucination Risk**: Ungrounded Token % (↓ Lower is Better) & Precision Source (% grounded in abstract)
3. **Inference Time**: Wall-Clock Latency per Sample (↓ Lower is Faster) & Throughput (tok/s ↑)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, gc, time, json

BASE_DIR = "/content/drive/MyDrive/JU_Benchmark_Project"
DATASET_DIR = os.path.join(BASE_DIR, "Dataset")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
MODEL_DIR = os.path.join(BASE_DIR, "checkpoints", "gemma-2-2b")

os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 128

print("Drive ready!")

In [ ]:
!pip -q install transformers accelerate bitsandbytes peft sentencepiece rouge-score

import torch

if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU: Runtime > Change runtime type > T4 GPU")

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import gc
import time
import json

BASE_DIR = "/content/drive/MyDrive/JU_Benchmark_Project"
DATASET_DIR = os.path.join(BASE_DIR, "Dataset")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
MODEL_CHECKPOINT_DIR = os.path.join(
    BASE_DIR, "checkpoints", "phi-3-mini"
)

os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODEL_CHECKPOINT_DIR, exist_ok=True)

MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 128

print("Google Drive mounted!")
print("Results:", RESULTS_DIR)
print("Input limit:", MAX_SOURCE_LENGTH)
print("Output limit:", MAX_TARGET_LENGTH)

In [ ]:
!pip -q install transformers accelerate bitsandbytes peft sentencepiece datasets rouge-score gdown

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU required! Go to Runtime > Change runtime type > T4 GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))
print(
    "VRAM:",
    round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2),
    "GB"
)

In [ ]:
import pandas as pd
import gdown

TRAIN_CSV = os.path.join(
    DATASET_DIR, "MixSub-SciHigh_train_FIRE.csv"
)

VAL_CSV = os.path.join(
    DATASET_DIR, "MixSub-SciHigh_val_FIRE.csv"
)

TEST_CSV = os.path.join(
    DATASET_DIR, "MixSub-SciHigh_test_FIRE(Masked).csv"
)

# Download missing train/validation datasets
downloads = {
    TRAIN_CSV: "1oydpEnDZ8Chu7dQpT5RGnnRTMvhfq-8Q",
    VAL_CSV: "1-A0tc0sFJGgwWEOywM9Ddyxi6G4K_Xhj"
}

for path, file_id in downloads.items():

    if not os.path.exists(path):
        print("Downloading:", os.path.basename(path))

        gdown.download(
            f"https://drive.google.com/uc?id={file_id}",
            path,
            quiet=False
        )

if not os.path.exists(VAL_CSV):
    raise FileNotFoundError(
        "Validation dataset missing. Check Google Drive."
    )

val_df = pd.read_csv(VAL_CSV)

val_df = val_df.dropna(
    subset=["Abstract", "Highlights"]
).reset_index(drop=True)

# Load masked test dataset if available
test_df = None

if os.path.exists(TEST_CSV):

    test_df = pd.read_csv(TEST_CSV)

    test_df["Abstract"] = (
        test_df["Abstract"].fillna("").astype(str)
    )

    print("Masked test dataset found:", len(test_df))

else:
    print("Masked test dataset unavailable. Test prediction skipped.")

print("\nDataset loaded successfully!")
print("Validation:", len(val_df))

In [ ]:

# ============================================================
# CELL 4 — FIXED PHI-3 MINI MODEL LOADING
# Native Transformers implementation
# ============================================================

import os
import gc
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"

MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 128

# Clear previous model objects
if "model" in globals():
    del model

gc.collect()
torch.cuda.empty_cache()

# GPU verification
assert torch.cuda.is_available(), "Enable GPU in Colab!"

print("GPU:", torch.cuda.get_device_name(0))

# 4-bit configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("\nLoading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=False,
    token=False
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

print("Tokenizer loaded!")

print("\nLoading native Phi-3 architecture...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=False,
    token=False,
    low_cpu_mem_usage=True
)

model.eval()

print("\n===================================")
print("PHI-3 MINI LOADED SUCCESSFULLY")
print("===================================")
print("Model:", MODEL_ID)
print("GPU:", torch.cuda.get_device_name(0))
print("Quantization: 4-bit NF4")
print("Implementation: Native Transformers")
print("===================================")


In [ ]:
import os
import time
import json
import torch
import numpy as np
import pandas as pd

MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 128

MAX_EVAL_SAMPLES = 5
MAX_TEST_SAMPLES = 5

assert "model" in globals()
assert "tokenizer" in globals()
assert "val_df" in globals()

os.makedirs(RESULTS_DIR, exist_ok=True)

print("Phi-3 benchmark ready!")